# 04-工具调用 - DeepSeek API

Function Calling 示例

In [2]:
from openai import OpenAI
import json
import os
from dotenv import load_dotenv

# 加载 .env 文件
load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("❌ 未找到 API Key")

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("VITE_DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
)

## 基础工具调用

In [2]:
# 定义工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取城市天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"}
                },
                "required": ["city"]
            }
        }
    }
]

# 调用
messages = [{"role": "user", "content": "北京天气如何？"}]
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    tools=tools
)

message = response.choices[0].message
if message.tool_calls:
    print(f"调用工具: {message.tool_calls[0].function.name}")
    print(f"参数: {message.tool_calls[0].function.arguments}")
else:
    print(f"直接回答: {message.content}")

调用工具: get_weather
参数: {"city": "北京"}


## 多轮工具调用测试

测试场景：模型先调用工具获取数据，然后基于工具结果生成自然语言回复。
这模拟了生产环境中的完整工具调用流程。

In [3]:
# 定义工具函数
def get_weather(city: str) -> str:
    """获取天气（模拟）"""
    return f"{city}今天晴朗，温度25°C"

def create_article(title: str, content: str) -> str:
    """创建文章（模拟）"""
    return f"文章《{title}》创建成功！字数：{len(content)}"

# 定义工具列表
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_article",
            "description": "创建一篇文章",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "文章标题"},
                    "content": {"type": "string", "description": "文章内容"}
                },
                "required": ["title", "content"]
            }
        }
    }
]

# ===== 完整的多轮工具调用流程 =====
messages = [
    {
        "role": "user",
        "content": "帮我查一下北京的天气，然后写一篇文章介绍今天北京的天气情况。"
    }
]

print("=== 第1步：检测工具调用 ===")
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    tools=tools
)

message = response.choices[0].message
print(f"AI响应类型: {message.role}")
print(f"是否有工具调用: {bool(message.tool_calls)}")

if message.tool_calls:
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    print(f"\n工具名称: {function_name}")
    print(f"工具参数: {arguments}")
    # 执行工具
    print(f"\n=== 第2步：执行工具 ===")
    if function_name == "get_weather":
        result = get_weather(**arguments)
    elif function_name == "create_article":
        result = create_article(**arguments)
    else:
        result = f"未知工具: {function_name}"
    print(f"工具执行结果: {result}")
    # 将工具调用和结果加入消息历史
    messages.append({
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "id": tool_call.id,
            "type": "function",
            "function": {
                "name": function_name,
                "arguments": tool_call.function.arguments
            }
        }]
    })
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    })
    # 第二步：获取自然语言回复
    print(f"\n=== 第3步：获取AI自然语言回复 ===")
    final_response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools
    )
    final_message = final_response.choices[0].message
    if final_message.tool_calls:
        # AI再次调用工具
        tool_call2 = final_message.tool_calls[0]
        function_name2 = tool_call2.function.name
        arguments2 = json.loads(tool_call2.function.arguments)
        print(f"AI再次调用工具: {function_name2}")
        print(f"参数: {arguments2}")
        if function_name2 == "create_article":
            result2 = create_article(**arguments2)
            print(f"工具执行结果: {result2}")
        # 再次发送结果给AI
        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": tool_call2.id,
                "type": "function",
                "function": {
                    "name": function_name2,
                    "arguments": tool_call2.function.arguments
                }
            }]
        })
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call2.id,
            "content": result2 if "result2" in dir() else "完成"
        })
        # 获取最终回复
        final_response2 = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages
        )
        print(f"\n=== 最终AI回复 ===")
        print(final_response2.choices[0].message.content)
    else:
        # AI直接回复
        print(f"\n=== 最终AI回复 ===")
        print(final_message.content)

=== 第1步：检测工具调用 ===
AI响应类型: assistant
是否有工具调用: True

工具名称: get_weather
工具参数: {'city': '北京'}

=== 第2步：执行工具 ===
工具执行结果: 北京今天晴朗，温度25°C

=== 第3步：获取AI自然语言回复 ===
AI再次调用工具: create_article
参数: {'title': '今日北京天气：晴朗宜人，温度舒适', 'content': '根据最新天气数据显示，北京今天迎来了一个晴朗的好天气。\n\n**天气概况：**\n- 天气状况：晴朗\n- 温度：25°C\n- 体感：舒适宜人\n\n**天气特点分析：**\n今天的北京天气十分理想，晴朗的天空为市民和游客提供了绝佳的户外活动条件。25°C的温度既不炎热也不寒冷，属于人体感觉最舒适的温度范围之一。\n\n**生活建议：**\n1. **户外活动**：适合进行散步、晨练、郊游等户外活动\n2. **穿着建议**：建议穿着轻薄的长袖或短袖衣物，早晚可适当添加薄外套\n3. **防晒措施**：虽然温度适宜，但晴朗天气下紫外线较强，建议做好防晒工作\n4. **空气质量**：晴朗天气通常伴随着较好的空气质量，适合开窗通风\n\n**旅游推荐：**\n对于来京旅游的朋友来说，今天是游览故宫、天坛、颐和园等景点的绝佳时机。晴朗的天气能让您更好地欣赏这些历史建筑的壮丽景色。\n\n**温馨提示：**\n虽然天气晴朗舒适，但北京昼夜温差可能较大，建议随身携带一件薄外套以备不时之需。同时，记得及时补充水分，保持身体水分平衡。\n\n总的来说，今天北京的天气条件非常优越，无论是工作、学习还是休闲娱乐，都是难得的好日子。希望大家能好好享受这个美好的天气！'}
工具执行结果: 文章《今日北京天气：晴朗宜人，温度舒适》创建成功！字数：512

=== 最终AI回复 ===
我已经为您查询了北京的天气情况，并撰写了一篇介绍今日北京天气的文章。

**查询结果：** 北京今天天气晴朗，温度25°C

**文章概述：** 我创建了一篇题为《今日北京天气：晴朗宜人，温度舒适》的文章，详细介绍了今天北京的天气特点。文章内容包括：
- 天气概况（晴朗，25°C）
- 天气特点分析
- 生活建议（穿着、防晒等）
- 旅游推荐
- 温馨提示

